In [6]:
!pip install pycocotools

In [7]:
from pycocotools.coco import COCO
import numpy as np
import skimage.io as io
import matplotlib.pyplot as plt
#import pylab
from torch import ConcreteModuleType

from torch.utils.data import DataLoader
from torchvision import datasets
from PIL import Image
import torch
import torchvision
import os


In [8]:
class myOwnDataset(torch.utils.data.Dataset):
    def __init__(self, root, annotation, transforms=None):
        self.root = root
        self.transforms = transforms
        self.coco = COCO(annotation)
        #self.ids = list(sorted(self.coco.imgs.keys()))
        self.catIds = self.coco.getCatIds(catNms=['person', 'dog', 'skateboard'])
        self.ids = self.coco.getImgIds(catIds=self.catIds)

    def __getitem__(self, index):
        # Own coco file
        coco = self.coco
        
        # Image ID
        img_id = self.ids[index]

        # List: get annotation id from coco
        ann_ids = coco.getAnnIds(imgIds=img_id)

        # Dictionary: target coco_annotation file for an image
        coco_annotation = coco.loadAnns(ann_ids)

        # path for input image
        path = coco.loadImgs(img_id)[0]['file_name']

        # open the input image
        img = Image.open(os.path.join(self.root, path))

        # number of objects in the image
        num_objs = len(coco_annotation)

        # Bounding boxes for objects
        # In coco format, bbox = [xmin, ymin, width, height]
        # In pytorch, the input should be [xmin, ymin, xmax, ymax]
        boxes = []
        for i in range(num_objs):
            xmin = coco_annotation[i]['bbox'][0]
            ymin = coco_annotation[i]['bbox'][1]
            xmax = xmin + coco_annotation[i]['bbox'][2]
            ymax = ymin + coco_annotation[i]['bbox'][3]
            boxes.append([xmin, ymin, xmax, ymax])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)

        # Labels (In my case, I only one class: target class or background)
        labels = torch.ones((num_objs,), dtype=torch.int64)

        # Tensorise img_id
        img_id = torch.tensor([img_id])

        # Size of bbox (Rectangular)
        areas = []
        for i in range(num_objs):
            areas.append(coco_annotation[i]['area'])

        areas = torch.as_tensor(areas, dtype=torch.float32)

        # Iscrowd
        iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

        # Annotation is in dictionary format
        my_annotation = {}
        my_annotation["boxes"] = boxes
        my_annotation["labels"] = labels
        my_annotation["image_id"] = img_id
        my_annotation["area"] = areas
        my_annotation["iscrowd"] = iscrowd

        if self.transforms is not None:
            img = self.transforms(img)

        return img, my_annotation

    def __len__(self):
        return len(self.ids)


# In my case, just added ToTensor
def get_transform():
    custom_transforms = []
    custom_transforms.append(torchvision.transforms.ToTensor())
    return torchvision.transforms.Compose(custom_transforms)

In [9]:
root = "../input/coco-2017-dataset/coco2017/train2017"
annotations_file = "../input/coco-2017-dataset/coco2017/annotations/instances_train2017.json"


In [10]:
my_dataset = myOwnDataset(root=root,
                          annotation=annotations_file,
                          transforms=get_transform()
                          )

# collate_fn needs for batch
def collate_fn(batch):
    return tuple(zip(*batch))

# Batch size
train_batch_size = 1

# own DataLoader
data_loader = torch.utils.data.DataLoader(my_dataset,
                                          batch_size=train_batch_size,
                                          shuffle=True,
                                          num_workers=4,
                                          collate_fn=collate_fn)

loading annotations into memory...


FileNotFoundError: [Errno 2] No such file or directory: '../input/coco-2017-dataset/coco2017/annotations/instances_train2017.json'

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# DataLoader is iterable over Dataset
# for imgs, annotations in data_loader:
#     imgs = list(img.to(device) for img in imgs)
#     annotations = [{k: v.to(device) for k, v in t.items()} for t in annotations]
#     print(annotations)

[{'boxes': tensor([[159.4900,  39.5600, 525.4500, 485.8900],
        [395.5500,   0.9700, 611.3800, 487.3100],
        [ 11.0500,   2.4200, 145.0600, 216.5500],
        [  0.0000,   3.9900,  54.8100, 251.2000],
        [134.7800, 358.5300, 459.3400, 586.8200],
        [278.8000,   1.6900, 325.8900,  57.0300],
        [283.4600,   0.0000, 364.4600,  54.6000]]), 'labels': tensor([1, 1, 1, 1, 1, 1, 1]), 'image_id': tensor([438915]), 'area': tensor([99190.0469, 62025.9375, 17018.4629,  7904.3652, 31716.5137,  1880.9768,
         1924.7651]), 'iscrowd': tensor([0, 0, 0, 0, 0, 0, 0])}]
[{'boxes': tensor([[287.6500, 243.0100, 573.8300, 438.3400],
        [ 97.0300,  56.0000, 196.1300, 400.7700],
        [242.0900,  83.1100, 253.3900, 115.4200],
        [447.1000,  73.2700, 453.6800,  89.0200],
        [295.7400, 387.9800, 448.8000, 435.4000]]), 'labels': tensor([1, 1, 1, 1, 1]), 'image_id': tensor([241837]), 'area': tensor([17771.8184, 21059.7246,   210.5607,    67.5695,  2752.4519]), 'iscrow